In [ ]:
import sys
import os

In [ ]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive
    in_colab = True

except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount("/content/drive")

    project_path = "/content/drive/My Drive/structure-loss-classification/"
    sys.path.append(project_path)

    my_local_data = "/content/drive/MyDrive/types/"

    os.chdir(project_path)
    %pip install -r requirements.txt

else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"

## Import Relevant Modules from Structure Loss Classification Package

In [ ]:
from lightning_modules.integrated import LitLeNet5, LitVGG16, LitResNet18
from utils.utils import get_category_names, load_targets, get_stat_metrics, load_hyperparameter, get_train_val_data
from train.train import train_model, train_with_cv
from visualization.display import display_metrics, process_plot_image
from datasets.datasets import CustomDatasetWrapper
from datasets.data_modules import CustomImageDataModule
from hyperparameter_tuning.tune import HyperParameterTuner

## Other Packages

In [ ]:
import torch
import torchvision.transforms as transforms
from ray import tune
import json

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, random_split

## Transformations

In [ ]:
toTensorAndNormalize = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((244, 244)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],  # mean
                         [0.5, 0.5, 0.5]  # std
                         )
])

## Define Classification Task

In [ ]:
classification_mode = "binary"

In [ ]:
task = {
    'binary': 2,
    'only_bad': 3,
    'all': 4
}

num_classes = task[classification_mode]

In [ ]:
dataset = CustomDatasetWrapper(
    root_dir=my_local_data,
    classification_mode=classification_mode,
    transform=toTensorAndNormalize,
)

In [ ]:
targets = dataset.get_all_labels() #load_targets(dataset)

In [ ]:
train_dataset, val_dataset = get_train_val_data(dataset, targets)

In [ ]:
datamodule = CustomImageDataModule(train_dataset, val_dataset)

In [ ]:
categories = get_category_names(dataset)
print(categories)

## Instantiate Lightning Model Class

In [ ]:
model_class = LitLeNet5

In [ ]:
model_class.__name__

## Training Configuration

In [ ]:
torch.set_float32_matmul_precision('highest')
trainer_config = {
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 200,
}
save_dir = f'logdir/pipeline_1/{model_class.__name__}/{classification_mode}/cv'

### Hyperparameter Tuning

In [ ]:
## environment variables for pretty display of results
%env RAY_AIR_NEW_OUTPUT=0
%env RAY_AIR_RICH_LAYOUT=0
%env RAY_AIR_FULL_TRACEBACKS=0

In [ ]:
## search space for hyperparameter tuning
search_space = {
   'model_params': {"num_classes": num_classes,
                    "size_layer_1": tune.choice([256, 128, 84]),
                    "size_layer_2": tune.choice([64, 32, 10])},

    'learning_rate': tune.loguniform(1e-5, 1e-3),
    'batch_size': tune.choice([32, 64, 128]),
    'num_workers': os.cpu_count()
}

In [ ]:
dirname = f"logdir/pipeline_1/{model_class.__name__}/"
filename = f"{dirname}/hyperparameters_{classification_mode}.json"

In [ ]:
!ls $dirname

In [ ]:
try:
    hyperparameters = load_hyperparameter(
            classification_mode = classification_mode,
            filename = filename
        )

except FileNotFoundError:

    my_tuner = HyperParameterTuner(model_class = model_class,
                                   datamodule = datamodule,
                                   search_space = search_space,
                                   num_samples = 10,
                                   num_epochs = 10)

    hyperparameters = my_tuner.hypertune()

    with open(filename, 'w') as f:
        json.dump(hyperparameters, f, indent=4)

### Training

In [ ]:
## k-fold cross-validation
n_splits = 5
kfold = StratifiedKFold(n_splits=n_splits, shuffle=True,)

In [ ]:
hyperparameters = {'model_params': {'num_classes': 2,
                                    'size_layer_1': 128,
                                    'size_layer_2': 64},
                   'learning_rate': 8.479282111072422e-05,
                   'batch_size': 16,
                   'num_workers': 8}

In [ ]:
save_dir

In [ ]:
%%capture
for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset, targets)):
    print(f"-------------- Fold: {fold + 1} --------------")

    train_data = torch.utils.data.Subset(dataset, train_idx)
    val_data = torch.utils.data.Subset(dataset, val_idx)

    data_module = CustomImageDataModule(train_dataset=train_data,
                                        val_dataset=val_data,
                                        batch_size=hyperparameters['batch_size'],
                                        num_workers=hyperparameters['num_workers'])

    model = LitLeNet5(params=hyperparameters)

    csv_logger = CSVLogger(f'{save_dir}/fold_{fold + 1}')

    trainer = pl.Trainer(**trainer_config,
                         logger=csv_logger,
                         enable_checkpointing=False,
                         )

    trainer.fit(model=model, datamodule=data_module)
    trainer.validate(model=model, datamodule=data_module)